# 02 — Graph Metrics & Day/Night Comparison

Explores the network topology for a set of contexts and compares
daytime vs. nighttime structure using the analysis module.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config import OUTPUT_ROOT
from analysis.day_night_comparison import run_comparison, build_node_comparison

YEAR  = 2016
PLOTS = ['MSTO', 'SPRA', 'LLOD']

## 1. Load node metrics for one context

In [ ]:
def load_nodes(year, plot, period):
    label = f'{year}_{plot}_{period}'
    path  = OUTPUT_ROOT / label / f'nodes_{label}.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df['period'] = period
    return df

# Load day + night for one plot
PLOT = 'SPRA'
nodes_d = load_nodes(YEAR, PLOT, 'daytime')
nodes_n = load_nodes(YEAR, PLOT, 'nighttime')
print('Daytime:', nodes_d.shape if nodes_d is not None else 'not found')
print('Nighttime:', nodes_n.shape if nodes_n is not None else 'not found')

In [ ]:
# Centrality distributions side-by-side
if nodes_d is not None and nodes_n is not None:
    metrics = ['degree', 'strength', 'betweenness', 'eigenvector', 'clustering']
    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 3))
    for ax, m in zip(axes, metrics):
        if m in nodes_d.columns:
            ax.hist(nodes_d[m], bins=20, alpha=0.6, label='day',   color='steelblue')
        if m in nodes_n.columns:
            ax.hist(nodes_n[m], bins=20, alpha=0.6, label='night', color='coral')
        ax.set_title(m)
        ax.legend(fontsize=8)
    plt.suptitle(f'{YEAR} {PLOT}', y=1.02)
    plt.tight_layout()
    plt.show()

## 2. Network position breakdown

In [ ]:
all_nodes = pd.concat(
    [n for n in [nodes_d, nodes_n] if n is not None],
    ignore_index=True,
)
if 'network_position' in all_nodes.columns:
    cross = pd.crosstab(all_nodes['period'], all_nodes['network_position'])
    cross.plot(kind='bar', figsize=(7, 4), colormap='Set2', edgecolor='black')
    plt.title(f'Network positions — {YEAR} {PLOT}')
    plt.ylabel('Bird count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 3. Day vs. Night structural comparison (all plots)

In [ ]:
summary, node_df = run_comparison(years=[YEAR], plots=PLOTS, save=False)
summary

In [ ]:
# Density comparison bar chart
if not summary.empty and 'day_edge_density' in summary.columns:
    x = range(len(summary))
    labels = summary['plot'].tolist()
    fig, ax = plt.subplots(figsize=(7, 4))
    w = 0.35
    ax.bar([i - w/2 for i in x], summary['day_edge_density'],   width=w, label='daytime',  color='steelblue')
    ax.bar([i + w/2 for i in x], summary['night_edge_density'], width=w, label='nighttime', color='coral')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels)
    ax.set_ylabel('Edge density')
    ax.set_title(f'Social network density — {YEAR}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Individual bird centrality: day vs. night

In [ ]:
# Shared birds — scatter degree day vs night
node_comp = build_node_comparison(YEAR, PLOT)
if not node_comp.empty and 'day_degree' in node_comp.columns:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    pairs = [('day_degree', 'night_degree', 'Degree'),
             ('day_betweenness', 'night_betweenness', 'Betweenness'),
             ('day_strength', 'night_strength', 'Strength')]
    for ax, (xc, yc, title) in zip(axes, pairs):
        if xc in node_comp.columns and yc in node_comp.columns:
            ax.scatter(node_comp[xc], node_comp[yc], alpha=0.5, s=20)
            lim = [0, max(node_comp[xc].max(), node_comp[yc].max()) * 1.05]
            ax.plot(lim, lim, 'r--', lw=1)
            ax.set_xlabel(f'Day {title}')
            ax.set_ylabel(f'Night {title}')
            ax.set_title(title)
    plt.suptitle(f'Day vs Night centrality — {YEAR} {PLOT}', y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Connector stability

In [ ]:
if not node_comp.empty and 'day_is_connector' in node_comp.columns:
    ct = pd.crosstab(
        node_comp['day_is_connector'].map({True: 'Day connector', False: 'Not day connector'}),
        node_comp['night_is_connector'].map({True: 'Night connector', False: 'Not night connector'}),
    )
    print('Connector overlap matrix:')
    print(ct)
    jaccard = summary['connector_jaccard'].iloc[0] if 'connector_jaccard' in summary.columns else 'N/A'
    print(f'Jaccard similarity of connector sets: {jaccard}')